# Exercise 3: Neural networks in PyTorch

In this exercise you’ll implement small neural-network building blocks from scratch and use them to train a simple classifier.

You’ll cover:
- **Basic layers**: Linear, Embedding, Dropout
- **Normalization**: LayerNorm and RMSNorm
- **MLPs + residual**: composing layers into deeper networks
- **Classification**: generating a learnable dataset, implementing cross-entropy from logits, and writing a minimal training loop

As before: fill in all `TODO`s without changing function names or signatures.
Use small sanity checks and compare to PyTorch reference implementations when useful.

In [1]:
from __future__ import annotations

import torch
from torch import nn

## Basic layers

In this section you’ll implement a few core layers that appear everywhere:

### `Linear`
A fully-connected layer that follows nn.Linear conventions:  
`y = x @ Wᵀ + b`

Important details:
- Parameters should be registered as `nn.Parameter`
- Store weight as (out_features, in_features) like nn.Linear.
- The forward pass should support leading batch dimensions: `x` can be shape `(..., in_features)`

### `Embedding`
An embedding table maps integer ids to vectors:
- input: token ids `idx` of shape `(...,)`
- output: vectors of shape `(..., embedding_dim)`

This is essentially a learnable lookup table.

### `Dropout`
Dropout randomly zeroes activations during training to reduce overfitting.
Implementation details:
- Only active in `model.train()` mode
- In training: drop with probability `p` and scale the kept values by `1/(1-p)` so the expected value stays the same
- In eval: return the input unchanged

## Instructions
- Do not use PyTorch reference modules for the parts you implement (e.g. don’t call nn.Linear inside your Linear).
- You may use standard tensor ops that you learned before (matmul, sum, mean, rsqrt, indexing, etc.).
- Use a parameter initialization method of your choice. We recommend something like Xavier-uniform.


In [2]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
        else:
            self.register_parameter("bias", None)

        bound = (6.0 / (in_features + out_features)) ** 0.5 #using xavier-uniform
        with torch.no_grad():
            self.weight.uniform_(-bound, bound)
            if self.bias is not None:
                self.bias.zero_()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (..., in_features)
        return: (..., out_features)
        """
        y = x @ self.weight.T
        if self.bias is not None:
            y = y + self.bias
        return y

linear = Linear(in_features=3, out_features=2)
linear_x = torch.ones(4, 3)
linear_out = linear(linear_x)
print(linear_out)
print(linear_out.shape)

tensor([[ 1.6669, -1.2628],
        [ 1.6669, -1.2628],
        [ 1.6669, -1.2628],
        [ 1.6669, -1.2628]], grad_fn=<AddBackward0>)
torch.Size([4, 2])


In [ ]:
class Embedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.weight = nn.Parameter(torch.empty(num_embeddings, embedding_dim))
        with torch.no_grad():
            self.weight.normal_(mean=0.0, std=1.0)
        #initialize vectors randomly from a normal distribution
    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """
        idx: (...,) int64
        return: (..., embedding_dim)
        """
        return self.weight[idx]

embedding = Embedding(num_embeddings=5, embedding_dim=3)
embedding_idx = torch.tensor([[0, 2], [3, 4]])
embedding_out = embedding(embedding_idx)
print(embedding_out)

tensor([[[-0.1933,  0.8217, -0.1422],
         [-0.2844,  0.5700, -0.5658]],

        [[-0.0715, -1.5041, -0.2081],
         [-1.0525,  0.4372, -1.6853]]], grad_fn=<IndexBackward0>)
torch.Size([2, 2, 3])


In [ ]:
class Dropout(nn.Module):
    def __init__(self, p: float):
        super().__init__()
        if p < 0 or p >= 1:
            raise ValueError("p must be in [0, 1)")
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        In train mode: drop with prob p and scale by 1/(1-p).
        In eval mode: return x unchanged.
        """
        if not self.training or self.p == 0:
            return x
        keep_mask = torch.rand_like(x) >= self.p #outputs a True false mask based on comparing the values of p and the random numbers
        return x * keep_mask.to(dtype=x.dtype) / (1 - self.p) #scaling to compensate on cutting off neurons
        #the scaling is applied to the output

dropout = Dropout(p=0.5)
dropout_x = torch.ones(8)
dropout.train()
print(dropout(dropout_x)) #dropout(dropout_x) = dropout.forward(dropout_x)
dropout.eval()
print(dropout(dropout_x))

tensor([0., 2., 2., 0., 2., 2., 0., 0.])
tensor([1., 1., 1., 1., 1., 1., 1., 1.])


## Normalization

Normalization layers help stabilize training by controlling activation statistics.

### LayerNorm
LayerNorm normalizes each example across its **feature dimension** (the last dimension):

- compute mean and variance over the last dimension
- normalize: `(x - mean) / sqrt(var + eps)`
- apply learnable per-feature scale and shift (`weight`, `bias`)

**In this exercise, assume `elementwise_affine=True` (always include `weight` and `bias`).**  
`weight` and `bias` each have shape `(D,)`.

LayerNorm is widely used in transformers because it does not depend on batch statistics.

### RMSNorm
RMSNorm is similar to LayerNorm but normalizes using only the root-mean-square:
- `x / sqrt(mean(x^2) + eps)` over the last dimension
- usually includes a learnable scale (`weight`)
- no mean subtraction

RMSNorm is popular in modern LLMs because it's faster.


In [5]:
class LayerNorm(nn.Module):
    def __init__(
        self, normalized_shape: int, eps: float = 1e-5, elementwise_affine: bool = True
    ):
        super().__init__()
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if elementwise_affine:
            self.weight = nn.Parameter(torch.ones(normalized_shape))
            self.bias = nn.Parameter(torch.zeros(normalized_shape))
        else:
            self.register_parameter("weight", None)
            self.register_parameter("bias", None)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Normalize over the last dimension.
        x: (..., D)
        """
        mean = x.mean(dim=-1, keepdim=True)
        var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)
        y = (x - mean) / torch.sqrt(var + self.eps)
        if self.elementwise_affine:
            y = y * self.weight + self.bias
        return y

layer_norm = LayerNorm(normalized_shape=3)
layer_norm_x = torch.tensor([[1.0, 2.0, 3.0], [2.0, 4.0, 6.0]])
layer_norm_out = layer_norm(layer_norm_x)
print(layer_norm_out)
print(layer_norm_out.mean(dim=-1))
print(layer_norm_out.var(dim=-1, unbiased=False))

tensor([[-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247]], grad_fn=<AddBackward0>)
tensor([0., 0.], grad_fn=<MeanBackward1>)
tensor([1.0000, 1.0000], grad_fn=<VarBackward0>)


In [6]:
class RMSNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-8):
        super().__init__()
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        RMSNorm: x / sqrt(mean(x^2) + eps) * weight
        over the last dimension.
        """
        rms = torch.sqrt((x ** 2).mean(dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight

rms_norm = RMSNorm(normalized_shape=3)
rms_norm_x = torch.tensor([[1.0, 2.0, 3.0], [2.0, 4.0, 6.0]])
rms_norm_out = rms_norm(rms_norm_x)
print(rms_norm_out)
print(torch.sqrt((rms_norm_out ** 2).mean(dim=-1)))

tensor([[0.4629, 0.9258, 1.3887],
        [0.4629, 0.9258, 1.3887]], grad_fn=<MulBackward0>)
tensor([1., 1.], grad_fn=<SqrtBackward0>)


## MLPs and residual networks

Now you’ll build larger networks by composing layers.

### MLP
An MLP is a stack of `depth` Linear layers with non-linear activations (use GELU) in between.
In this exercise you’ll support:
- configurable depth
- a hidden dimension
- optional LayerNorm between layers (a common stabilization trick)

A key skill is building networks using `nn.ModuleList` / `nn.Sequential` while keeping shapes consistent.

### Transformer-style FeedForward (FFN)
A transformer block contains a position-wise feedforward network:
- `D -> 4D -> D` (by default)
- activation is typically **GELU**

This is essentially an MLP applied independently at each token position.

### Residual wrapper
Residual connections are the simplest form of “skip connection”:
- output is `x + fn(x)`

They improve gradient flow and allow training deeper networks more reliably.

In [7]:
class MLP(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        out_dim: int,
        depth: int,
        use_layernorm: bool = False,
    ):
        super().__init__()
        if depth < 1:
            raise ValueError("depth must be at least 1")

        dims = [in_dim] + [hidden_dim] * (depth - 1) + [out_dim]
        layers = []
        for i in range(depth):
            layers.append(Linear(dims[i], dims[i + 1]))
            if i < depth - 1:
                if use_layernorm:
                    layers.append(LayerNorm(dims[i + 1]))
                layers.append(nn.GELU())
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

mlp = MLP(in_dim=3, hidden_dim=5, out_dim=2, depth=3, use_layernorm=True)
mlp_x = torch.randn(4, 3)
mlp_out = mlp(mlp_x)
print(mlp_out)
print(mlp_out.shape)

tensor([[ 0.9317,  0.7618],
        [ 0.6150,  0.5049],
        [-0.1494,  1.1482],
        [-0.1912,  0.6607]], grad_fn=<AddBackward0>)
torch.Size([4, 2])


In [9]:
class FeedForward(nn.Module):
    """
    Transformer-style FFN: D -> 4D -> D (default)
    """

    def __init__(self, d_model: int, d_ff: int | None = None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = Linear(d_model, d_ff)
        self.act = nn.GELU()
        self.fc2 = Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.act(self.fc1(x)))

ffn = FeedForward(d_model=3, d_ff=6)
ffn_x = torch.randn(2, 4, 3)
ffn_out = ffn(ffn_x)
print(ffn_x)
print(ffn_out)
print(ffn_out.shape)

tensor([[[ 1.2673, -0.9192, -0.0883],
         [ 1.0161, -0.5100, -1.2000],
         [ 1.4013, -0.0510, -1.4589],
         [ 2.2608,  0.4911,  1.1107]],

        [[-0.1895,  0.7541,  0.7660],
         [-0.3143,  0.0559, -0.5191],
         [-0.8268,  2.1698,  1.5188],
         [ 0.7041,  0.6948, -0.3684]]])
tensor([[[ 2.2524e-01,  1.0437e-01, -5.5636e-01],
         [ 1.5815e-01, -1.2997e-01, -4.8313e-01],
         [ 1.1380e-04, -1.7586e-01, -6.6095e-01],
         [-3.2093e-02, -1.5130e-01, -6.2562e-01]],

        [[-1.4962e-01, -9.7796e-03, -1.0055e-02],
         [-5.2829e-02,  1.3822e-01,  1.8962e-01],
         [-2.7699e-01,  1.9759e-01,  8.3783e-02],
         [-1.0788e-01, -1.9237e-01, -1.2670e-01]]], grad_fn=<AddBackward0>)
torch.Size([2, 4, 3])


In [10]:
class Residual(nn.Module):
    def __init__(self, fn: nn.Module):
        super().__init__()
        self.fn = fn

    def forward(self, x: torch.Tensor, *args, **kwargs) -> torch.Tensor:
        return x + self.fn(x, *args, **kwargs)

residual = Residual(FeedForward(d_model=3, d_ff=6))
residual_x = torch.randn(2, 4, 3)
residual_out = residual(residual_x)
print(residual_out)
print(residual_out.shape)

tensor([[[-0.0272, -0.2138,  0.4151],
         [-0.7160, -0.0703,  0.7899],
         [-1.4024, -0.0240, -0.7987],
         [ 0.7466,  0.6448,  0.4498]],

        [[-0.2964, -2.3437,  2.2716],
         [ 0.2567,  0.9383, -0.0766],
         [-2.1601, -2.2044,  0.5510],
         [-0.1168, -1.8034, -0.5596]]], grad_fn=<AddBackward0>)
torch.Size([2, 4, 3])


## Classification problem

In this section you’ll put everything together in a minimal MNIST classification experiment.

You will:
1) download and load the MNIST dataset
2) implement cross-entropy from logits (stable, using log-softmax)
3) build a simple MLP-based classifier (flatten MNIST images first)
4) write a minimal training loop
5) report train loss curve and final accuracy

The goal here is not to reach state-of-the-art accuracy, but to understand the full pipeline:
data → model → logits → loss → gradients → parameter update.

### Model notes
- We want you to combine the MLP we implemented above with the classification head we define below into one model 

### MNIST notes
- MNIST images are `28×28` grayscale.
- After `ToTensor()`, each image has shape `(1, 28, 28)` and values in `[0, 1]`.
- For an MLP classifier, we flatten to a vector of length `784`.

## Deliverables
- Include a plot of your train loss curve in the video submission as well as a final accuracy. 
- **NOTE** Here we don't grade on model performance but we expect you to achieve at least 70% accuracy to confirm a correct model implementation.

In [12]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [13]:
transform = transforms.ToTensor()  # -> float32 in [0,1], shape (1, 28, 28)

train_ds = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)

print(len(train_loader), len(test_loader))

100.0%
100.0%
100.0%
100.0%


469 40


In [14]:
def cross_entropy_from_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
) -> torch.Tensor:
    """
    Compute mean cross-entropy loss from logits.

    logits: (B, C)
    targets: (B,) int64

    Requirements:
    - Use log-softmax for stability (do not use torch.nn.CrossEntropyLoss, we check this in the autograder).
    """
    log_probs = logits - torch.logsumexp(logits, dim=-1, keepdim=True)
    batch_indices = torch.arange(targets.shape[0], device=targets.device)
    return -log_probs[batch_indices, targets].mean()

ce_logits = torch.tensor([[2.0, 0.0, 1.0], [0.0, 3.0, 1.0]])
ce_targets = torch.tensor([0, 1])
ce_loss = cross_entropy_from_logits(ce_logits, ce_targets)
print(ce_loss)

tensor(0.2887)


In [15]:
class ClassificationHead(nn.Module):
    def __init__(self, d_in: int, num_classes: int):
        super().__init__()
        self.proj = Linear(d_in, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (..., d_in)
        return: (..., num_classes) logits
        """
        return self.proj(x)

head = ClassificationHead(d_in=4, num_classes=10)
head_x = torch.randn(3, 4)
head_logits = head(head_x)
print(head_logits)
print(head_logits.shape)

tensor([[-0.2183,  0.0286,  0.4767, -0.4543,  0.2412, -0.1465, -0.3865, -0.5977,
          0.1814,  0.3598],
        [ 0.6430,  0.5667,  0.9882, -0.6915, -0.0431, -0.3446,  0.9264, -0.3986,
         -1.0556,  0.3125],
        [-1.1339,  0.4402,  0.3432, -0.7378,  0.1298,  0.4932, -0.6355, -0.9163,
          0.4102,  0.0070]], grad_fn=<AddBackward0>)
torch.Size([3, 10])


In [19]:
def accuracy(loader):
    correct = 0
    total = 0
    model.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            preds = logits.argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.numel()
    return correct / total

In [17]:
def train_classifier(
    model: nn.Module,
    train_data_loader: DataLoader,
    test_data_loader: DataLoader,
    lr: float,
    epochs: int,
    seed: int = 0,
) -> list[float]:
    """
    Minimal training loop for MNIST classification.

    Steps:
    - define optimizer
    - for each epoch:
        - sample minibatches
        - forward -> cross-entropy -> backward -> optimizer step
      - compute test accuracy at the end of each epoch
    - return list of training losses (one per update step)

    Requirements:
    - call model.train() during training and model.eval() during evaluation
    - do not use torch.nn.CrossEntropyLoss (use your cross_entropy_from_logits)
    """
    torch.manual_seed(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []

    for epoch in range(epochs):
        model.train()
        for x, y in train_data_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = cross_entropy_from_logits(logits, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for x, y in test_data_loader:
                x = x.to(device)
                y = y.to(device)
                logits = model(x)
                preds = logits.argmax(dim=-1)
                correct += (preds == y).sum().item()
                total += y.numel()
        print(f"epoch {epoch + 1}: test accuracy = {correct / total:.4f}")

    return losses


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nn.Sequential(
    nn.Flatten(),
    MLP(in_dim=28 * 28, hidden_dim=256, out_dim=128, depth=2, use_layernorm=True),
    ClassificationHead(d_in=128, num_classes=10),
).to(device)

print(model)
losses = train_classifier(model, train_loader, test_loader, lr=1e-3, epochs=3)
print(accuracy(test_loader))

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): MLP(
    (net): Sequential(
      (0): Linear()
      (1): LayerNorm()
      (2): GELU(approximate='none')
      (3): Linear()
    )
  )
  (2): ClassificationHead(
    (proj): Linear()
  )
)
epoch 1: test accuracy = 0.9648
epoch 2: test accuracy = 0.9734
epoch 3: test accuracy = 0.9717
0.9717
